In [5]:
import re
import pandas as pd
import requests
import yaml
import time
import random
import uuid
import os

In [6]:
# Load credentials from creds.yaml
with open("../creds.yaml", "r") as stream:
    try:
        creds = yaml.safe_load(stream)
    except yaml.YAMLError as e:
        print(f"Error loading YAML file: {e}")

# initialize SocialCrawl API key to use for requests
socialcrawl_key = creds["socialcrawl"]["api_key"]

In [8]:
# Used to test the SocialCrawl API for Instagram profile data
resInstagram = requests.get(
    "https://www.socialcrawl.dev/v1/instagram/profile",
    params={"handle": "instagram"},
    headers={"x-api-key": socialcrawl_key},
    timeout=15,
)

# Make sure the request was successful
print("Instagram API Status: ", resInstagram.status_code)
print("Instagram API Response: ", resInstagram.text[:500])

Instagram API Status:  200
Instagram API Response:  {"success":true,"platform":"instagram","endpoint":"/v1/instagram/profile","data":{"author":{"id":"25025320","username":"instagram","display_name":"Instagram","avatar_url":"https://scontent-mia5-2.cdninstagram.com/v/t51.82787-19/550891366_18667771684001321_1383210656577177067_n.jpg?_nc_cat=1&ig_cache_key=GGbv1SApFvlLPVJCAOtF0XrDJjITbmNDAQAB0j-ccb7-5&ccb=7-5&_nc_sid=669407&efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLnd3dy4xMDgwLkMzIn0%3D&_nc_ohc=gTtzo3PzZjwQ7kNvwGi8zpL&_nc_oc=AdpO_eIfiSIcvRl9bShrVcjj


In [9]:
# Downloading recent posts from Instagram (SocialCrawl gives 100 credits TOTAL)
def get_posts_instagram(handle, max_pages=3):
    """ 
    Helper function to get posts from an Instagram account using the SocialCrawl API.
    Args:
        handle (str): Instagram username to scrape posts from.
        max_pages (int): Maximum number of pages to scrape. Each page contains up to 12 posts. Default is 3 pages (36 posts).
    Returns:
        list: A list of posts retrieved from the Instagram account.
    """
    rows, cursor = [], None
    for _ in range(max_pages):
        params = {"handle": handle}
        if cursor:
            params["next_max_id"] = cursor
        res = requests.get(
            "https://www.socialcrawl.dev/v1/instagram/profile/posts",
            params=params, headers={"x-api-key": socialcrawl_key}, timeout=30,
        )
        res.raise_for_status()
        body = res.json()
        data = body["data"]
        if len(rows) == 0:
            print(body.get("credits_remaining"))
            print(str(data)[:1500])
        items = data if isinstance(data, list) else (data.get("posts") or data.get("items") or [])
        rows.extend(items)
        cursor = (data.get("next_max_id") if isinstance(data, dict) else None) or body.get("next_max_id")
        if not cursor:
            break
        time.sleep(random.uniform(1, 3))
    return rows


In [11]:
# Import club and university information from the spreadsheet we created
social_media = pd.read_csv("../Data/Social Media for University - data.csv")
social_media.columns = social_media.columns.str.strip()

for column in ["university", "club", "instagram_handle", "facebook_handle", "instagram_link", "facebook_link"]:
    social_media[column] = social_media[column].fillna("").astype(str).str.strip()

In [12]:
# Focusing on instagram pages:
instagram_rows = social_media[
    social_media["instagram_handle"].ne("")
    & social_media["instagram_link"].fillna("").str.contains("instagram.com", case=False, regex=False)
]

# Instagram hangles & their club and university information
club = dict(zip(instagram_rows["instagram_handle"], instagram_rows["club"]))
university = dict(zip(instagram_rows["instagram_handle"], instagram_rows["university"]))
instagram_handle = instagram_rows["instagram_handle"].drop_duplicates().tolist()

print("Instagram handles loaded:", instagram_handle)

raw_by_handle_instagram = {
    handle: get_posts_instagram(handle)
    for handle in instagram_handle
}

Instagram handles loaded: ['ju.ieee', 'diu.cpc', 'diu_robotics_club', 'aiubfinanceclub', 'aiubenglish.club', 'aiubcomputerclub', 'aiubpcofficial', 'aiubfilmclub_official', 'aiubesportsclub']
88
{'items': [{'post': {'id': '3208589558413751241', 'url': 'https://www.instagram.com/p/CyHMzQbR3PJ/', 'content': {'text': None, 'media_urls': 'https://scontent-atl3-1.cdninstagram.com/v/t51.82787-15/628065112_18360617242164744_448216875388839578_n.jpg?stp=dst-jpg_e35_tt6&_nc_cat=110&_nc_map=urlgen_bucketless&ig_cache_key=MzIwODU4OTU1ODQxMzc1MTI0MQ%3D%3D.3-ccb7-5&ccb=7-5&_nc_sid=58cdad&efg=eyJ2ZW5jb2RlX3RhZyI6IkZFRUQueHBpZHMuMzk5LnNkci5yZWd1bGFyX3Bob3RvLkMzIn0%3D&_nc_ohc=YU4uEQRAd2AQ7kNvwHpiIQg&_nc_oc=AdpSRtePBH_pLTNOqUF_bRQ2aMANR_2AsSV2T6L2ZZAMg1f4056rTS_GEfrjhw98jDI&_nc_ad=z-m&_nc_cid=0&_nc_zt=23&_nc_ht=scontent-atl3-1.cdninstagram.com&_nc_gid=Qn-f0z7bzxlD79TAxRJYVA&_nc_ss=7339b&_nc_ct=RQosPBUIFQIlBAAAAA&oh=00_AQSfzO1ERp8omftZoYPMyZEbTm5xpEfa8IRw147MTUW1bw&oe=6B09A1B4', 'thumbnail_url': None, 'd

In [13]:
# Dictionaries: words/phrases that signal each concept (classifier)
ai_terms = [
    "ai", "a.i.", "artificial intelligence", "machine learning",
    "generative ai", "genai", "chatgpt", "gemini", "claude", "llm",
    "deep learning", "neural network", "data science", "prompt engineering",
]
event_terms = [
    "event", "seminar", "workshop", "conference", "symposium", "webinar",
    "talk", "meetup", "bootcamp", "hackathon", "summit", "masterclass",
    "competition", "register", "join us",
]

In [14]:
def make_pattern(terms):
    """Match whole words only, so 'ai' does NOT match 'said', 'maintain', 'email'.
        Args: terms (list): A list of terms to match.
        Returns: re.Pattern: A compiled regular expression pattern.
    """
    return re.compile(r"\b(" + "|".join(re.escape(t) for t in terms) + r")\b", re.IGNORECASE)

ai_pattern = make_pattern(ai_terms)
event_pattern = make_pattern(event_terms)

In [15]:
def classify_post(caption, instagram_handle="", facebook_handle=""):
    """ Classify a post to see if it's related to AI or an event using the dictionaries of terms (ai_terms and event_terms) 
        Args: caption (str): The caption text of the Instagram post to classify.
        instagram_handle (str): The Instagram handle of the user who posted the content.
        facebook_handle (str): The Facebook handle of the user who posted the content.
        Returns: dict: A dictionary containing classification results, including whether the post is related to AI, an event, both, 
        and the specific matches found.
    """
    text = re.sub(r"\s+", " ", str(caption or ""))
    ai_hits = sorted({m.lower() for m in ai_pattern.findall(text)})
    event_hits = sorted({m.lower() for m in event_pattern.findall(text)})
    return {
        "is_ai": len(ai_hits) > 0,
        "is_event": len(event_hits) > 0,
        "ai_event": len(ai_hits) > 0 and len(event_hits) > 0,
        "ai_matches": ai_hits,
        "event_matches": event_hits,
        "instagram_handle": instagram_handle,
        "facebook_handle": facebook_handle,
    }

In [20]:
def find_caption(item):
    """ Helper function to extract the caption text from a post item. 
        Args: item (dict): The post item from which to extract the caption.
        Returns: str: The caption text, or an empty string if not found.
    """
    post = item.get("post", item)
    content = post.get("content") or {}
    return content.get("text") or ""

rows = []
for handle, raw in raw_by_handle_instagram.items():
    for item in raw:
        rows.append({
            "instagram_handle": handle,
            "url": item.get("post", item).get("url"),
            "caption": find_caption(item),
        })

posts = pd.DataFrame(rows)
classified = posts.apply(
    lambda row: classify_post(
        row["caption"], 
        instagram_handle=row["instagram_handle"]
    ),
    axis=1,
).apply(pd.Series)
posts = pd.concat([posts, classified], axis=1)

OUT = "../Data/bangladesh_universities_ai_promotion_classified.csv"

#posts["club"] = posts["club_name"]

# doing this to avoid duplicates if the script is run multiple times
if os.path.exists(OUT):
    existing = pd.read_csv(OUT)
    existing_urls = set(existing["url"].dropna())
    posts = posts[~posts.url.isin(pd.read_csv(OUT).url)]

In [21]:
# appending new posts to the csv
posts.to_csv(OUT, 
             mode="a", 
             header=not os.path.exists(OUT), 
             index=False,
             )
print(len(posts), "new posts added")
posts.head(10)

8 new posts added


,instagram_handle,url,caption,is_ai,is_event,ai_event,ai_matches,event_matches,instagram_handle,facebook_handle
36,aiubfinanceclub,https://www.instagram.com/p/DgAuBwvzVBN/,The charts are speaking. It’s time to understa...,False,False,False,[],[],aiubfinanceclub,
37,aiubfinanceclub,https://www.instagram.com/p/DgARWKQzwiV/,The numbers are there. The patterns are there....,False,False,False,[],[],aiubfinanceclub,
38,aiubfinanceclub,https://www.instagram.com/p/DgAS3wMTfod/,"GTA called, but we’re busy checking the share ...",False,False,False,[],[],aiubfinanceclub,
40,aiubfinanceclub,https://www.instagram.com/p/DgARi19zSau/,POV: MSI Route to Market actually understood t...,False,False,False,[],[],aiubfinanceclub,
84,aiubfilmclub_official,https://www.instagram.com/p/DgCTR7HjO1M/,🎬 THE WAIT IS OVER! OFFICIAL RECRUITMENT RESUL...,False,False,False,[],[],aiubfilmclub_official,
96,aiubesportsclub,https://www.instagram.com/p/DgCMuzah17U/,WE ARE LIVE! \n\nJoin us at the D Building Lob...,False,True,False,[],[join us],aiubesportsclub,
97,aiubesportsclub,https://www.instagram.com/p/DgCGdeihHUY/,AIUB ESPORTS CLUB PRESENTS: WELCOME TO VICE CI...,False,True,False,[],"[event, register]",aiubesportsclub,
98,aiubesportsclub,https://www.instagram.com/p/DgA1ulWTujt/,"WELCOME TO VICE CITY, AIUB!\n\nAIUB eSports Cl...",False,False,False,[],[],aiubesportsclub,
